# Diabetes Detection from ECG — Sym4 + EMD (No ICA)

This notebook implements a clean ECG diabetes-classification pipeline using **Symlet-4 (`sym4`) wavelet denoising** followed by **Empirical Mode Decomposition (EMD)** for feature extraction.

### Final pipeline

**Raw ECG → Sym4 denoising → Z-score normalization → R-peak detection → fixed heartbeat windows → EMD → IMF/residue features → grouped train/test split → classifier → Normal / Diabetes**

ICA has been removed completely. The classifier receives only EMD-derived features plus the original beat-window standard deviation feature.

A grouped split by ECG record is retained so heartbeat windows from the same record cannot appear in both training and testing.


In [ ]:
# Mount Google Drive (Google Colab)
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

# Required packages
!pip install PyWavelets EMD-signal scikit-learn matplotlib pandas seaborn scipy joblib --quiet


In [ ]:
import os
from glob import glob
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from scipy import stats
from scipy.signal import find_peaks

import pywt
from PyEMD import EMD
from joblib import Parallel, delayed

from sklearn.model_selection import StratifiedGroupKFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
)

warnings.filterwarnings('ignore')
np.random.seed(42)


## 1. Configuration

The original heartbeat-window settings are retained. `MAX_IMFS = 5` keeps the EMD representation fixed in size and controls computational cost.


In [ ]:
# Beat-window settings
window_before = 90
window_after = 120
EXPECTED_WINDOW_LENGTH = window_before + window_after

# EMD settings
MAX_IMFS = 5
EMD_N_JOBS = -1      # -1 = use all CPU cores; use 1 if Colab RAM is limited
RANDOM_STATE = 42

# Google Drive dataset paths
diabetes_dir = '/content/drive/MyDrive/datasets_v2/diabetes'
normal_dir   = '/content/drive/MyDrive/datasets_v2/normal'

# Cache for expensive Sym4 + EMD feature extraction.
# Delete this file if you change the dataset, window size, Sym4 settings, or EMD settings.
EMD_CACHE_PATH = '/content/drive/MyDrive/datasets_v2/emd_features_sym4_max5.pkl'

print('Configuration loaded.')


## 2. Sym4 wavelet denoising

Sym4 is used only for **noise reduction**. EMD is applied later to the already denoised heartbeat windows for adaptive decomposition and feature extraction.


In [ ]:
def denoise_sym4(data, threshold_factor=0.04):
    """Denoise a 1-D ECG signal using the existing Symlet-4 wavelet approach."""
    x = np.asarray(data, dtype=np.float64)

    if x.ndim != 1 or len(x) < 2:
        return x.copy()

    w = pywt.Wavelet('sym4')
    maxlev = pywt.dwt_max_level(len(x), w.dec_len)

    if maxlev <= 0:
        return x.copy()

    coeffs = pywt.wavedec(x, 'sym4', level=maxlev)

    for i in range(1, len(coeffs)):
        scale = np.max(np.abs(coeffs[i]))
        if scale > 0:
            coeffs[i] = pywt.threshold(
                coeffs[i],
                threshold_factor * scale,
                mode='soft'
            )

    reconstructed = pywt.waverec(coeffs, 'sym4')
    return reconstructed[:len(x)]


def safe_zscore(x):
    x = np.asarray(x, dtype=np.float64)
    std = np.std(x)
    if not np.isfinite(std) or std < 1e-12:
        return np.zeros_like(x)
    return (x - np.mean(x)) / std

## 3. Load ECG records, denoise them, detect R-peaks, and create fixed heartbeat windows

Only complete windows of length `window_before + window_after = 210` samples are retained. Each window keeps its `record_id` so the train/test split can be performed at ECG-record level rather than beat level.


In [ ]:
def process_ecg_folder(folder_path, label, column_name):
    print(f'\nProcessing folder: {folder_path}')
    csv_files = sorted(glob(os.path.join(folder_path, '*.csv')))
    print(f'Found {len(csv_files)} CSV files.')

    results = []

    for i, file in enumerate(csv_files, 1):
        raw_df = pd.read_csv(file)

        if column_name not in raw_df.columns:
            print(f'Skipping {file}: column {column_name!r} not found')
            continue

        # Make the input numeric and fill occasional missing samples safely.
        series = pd.to_numeric(raw_df[column_name], errors='coerce')
        series = series.interpolate(limit_direction='both').ffill().bfill()
        ecg_signal = series.to_numpy(dtype=np.float64)

        if len(ecg_signal) < EXPECTED_WINDOW_LENGTH:
            print(f'Skipping {file}: signal too short')
            continue

        denoised_signal = denoise_sym4(ecg_signal)
        normalized_signal = safe_zscore(denoised_signal)

        # Retained from the original notebook.
        # Ideally, tune `height` and `distance` using the actual sampling rate.
        r_peaks, _ = find_peaks(
            normalized_signal,
            height=0.5,
            distance=150
        )

        kept = 0
        basename = os.path.basename(file)
        record_id = f'{label}_{basename}'

        for r_peak in r_peaks:
            start = int(r_peak - window_before)
            end   = int(r_peak + window_after)

            # Keep only complete, fixed-length beat windows.
            if start < 0 or end > len(normalized_signal):
                continue

            window_signal = normalized_signal[start:end]
            if len(window_signal) != EXPECTED_WINDOW_LENGTH:
                continue

            results.append({
                'record_id': record_id,
                'file': basename,
                'r_peak': int(r_peak),
                'window_signal': window_signal.astype(np.float32),
                'standard_deviation': float(np.std(window_signal)),
                'diabetes': int(label),
            })
            kept += 1

        print(
            f'Processed {i}/{len(csv_files)} files | '
            f'R-peaks: {len(r_peaks)} | complete windows kept: {kept} | {basename}'
        )

    out = pd.DataFrame(results)
    if out.empty:
        return out

    out = out.dropna(subset=['standard_deviation']).reset_index(drop=True)
    return out


# Original column names are preserved.
diabetes_df = process_ecg_folder(
    diabetes_dir,
    label=1,
    column_name='day'
)

normal_df = process_ecg_folder(
    normal_dir,
    label=0,
    column_name='filtered'
)

# Combine classes
df = pd.concat([diabetes_df, normal_df], ignore_index=True)

print('\nCombined dataset shape:', df.shape)
print('Unique ECG records:', df['record_id'].nunique())
print('\nClass distribution by beat window:')
print(df['diabetes'].value_counts())
print('\nClass distribution by ECG record:')
print(df.groupby('record_id')['diabetes'].first().value_counts())

## 4. Empirical Mode Decomposition (EMD)

Each **Sym4-denoised and normalized heartbeat window** is adaptively decomposed into Intrinsic Mode Functions (IMFs) and a residue.

For each of the first five IMFs, the notebook extracts:

- energy
- relative energy
- RMS
- standard deviation
- mean absolute amplitude
- normalized energy entropy
- skewness
- kurtosis
- zero-crossing rate

Two residue features are also extracted: residue energy and residue standard deviation.


In [ ]:
EPS = 1e-12


def normalized_energy_entropy(x):
    """Entropy of normalized squared amplitudes, normalized to approximately [0, 1]."""
    x = np.asarray(x, dtype=np.float64)
    power = x * x
    total = np.sum(power)

    if total <= EPS or len(x) <= 1:
        return 0.0

    p = power / total
    entropy = -np.sum(p * np.log2(p + EPS))
    return float(entropy / np.log2(len(x)))


def zero_crossing_rate(x):
    x = np.asarray(x, dtype=np.float64)
    if len(x) < 2:
        return 0.0
    return float(np.mean(np.signbit(x[1:]) != np.signbit(x[:-1])))


def _safe_shape_stats(x):
    x = np.asarray(x, dtype=np.float64)
    s = np.std(x)
    if not np.isfinite(s) or s < EPS:
        return 0.0, 0.0
    return float(stats.skew(x, bias=False)), float(stats.kurtosis(x, fisher=True, bias=False))


def emd_feature_dict(signal, max_imfs=MAX_IMFS):
    """Return a fixed-length dictionary of EMD/IMF features for one ECG beat."""
    x = np.asarray(signal, dtype=np.float64)
    signal_energy = float(np.sum(x * x)) + EPS

    emd = EMD()

    try:
        # Execute the decomposition, then explicitly separate IMFs from the residue.
        emd.emd(x, max_imf=max_imfs)
        imfs, residue = emd.get_imfs_and_residue()
    except Exception:
        # Keep the pipeline robust to pathological/near-constant windows.
        imfs = np.empty((0, len(x)), dtype=np.float64)
        residue = x.copy()

    features = {}

    for i in range(max_imfs):
        prefix = f'emd_imf{i+1}'

        if i < len(imfs):
            imf = np.asarray(imfs[i], dtype=np.float64)
            skewness, kurt = _safe_shape_stats(imf)

            features[f'{prefix}_energy'] = float(np.mean(imf ** 2))
            features[f'{prefix}_relative_energy'] = float(np.sum(imf ** 2) / signal_energy)
            features[f'{prefix}_rms'] = float(np.sqrt(np.mean(imf ** 2)))
            features[f'{prefix}_std'] = float(np.std(imf))
            features[f'{prefix}_mean_abs'] = float(np.mean(np.abs(imf)))
            features[f'{prefix}_entropy'] = normalized_energy_entropy(imf)
            features[f'{prefix}_skewness'] = skewness
            features[f'{prefix}_kurtosis'] = kurt
            features[f'{prefix}_zcr'] = zero_crossing_rate(imf)
        else:
            # Pad missing IMFs so that all rows have identical feature dimensions.
            features[f'{prefix}_energy'] = 0.0
            features[f'{prefix}_relative_energy'] = 0.0
            features[f'{prefix}_rms'] = 0.0
            features[f'{prefix}_std'] = 0.0
            features[f'{prefix}_mean_abs'] = 0.0
            features[f'{prefix}_entropy'] = 0.0
            features[f'{prefix}_skewness'] = 0.0
            features[f'{prefix}_kurtosis'] = 0.0
            features[f'{prefix}_zcr'] = 0.0

    residue = np.asarray(residue, dtype=np.float64)
    features['emd_residue_energy'] = float(np.mean(residue ** 2))
    features['emd_residue_std'] = float(np.std(residue))

    return features

### Visual check of EMD on one heartbeat

This plot lets you inspect the Sym4-denoised beat, its IMFs, and the residue before extracting features for the full dataset.


In [ ]:
def plot_emd_example(signal, max_imfs=MAX_IMFS):
    x = np.asarray(signal, dtype=np.float64)
    emd = EMD()
    emd.emd(x, max_imf=max_imfs)
    imfs, residue = emd.get_imfs_and_residue()

    n_rows = 2 + len(imfs)
    fig, axes = plt.subplots(n_rows, 1, figsize=(12, 2.0 * n_rows), sharex=True)

    axes[0].plot(x)
    axes[0].set_title('Sym4-denoised, normalized ECG beat')

    for i, imf in enumerate(imfs):
        axes[i + 1].plot(imf)
        axes[i + 1].set_ylabel(f'IMF {i+1}')

    axes[-1].plot(residue)
    axes[-1].set_ylabel('Residue')
    axes[-1].set_xlabel('Sample')

    plt.tight_layout()
    plt.show()


if len(df) > 0:
    plot_emd_example(df.iloc[0]['window_signal'])

## 5. Extract EMD features for all heartbeat windows

This is the computationally expensive step. The feature table is cached so repeated model experiments do not have to rerun EMD.


In [ ]:
def extract_all_emd_features(dataframe, n_jobs=EMD_N_JOBS):
    signals = dataframe['window_signal'].tolist()

    if n_jobs == 1:
        rows = [emd_feature_dict(signal) for signal in signals]
    else:
        rows = Parallel(
            n_jobs=n_jobs,
            backend='loky',
            batch_size=64,
            verbose=10,
        )(
            delayed(emd_feature_dict)(signal)
            for signal in signals
        )

    return pd.DataFrame(rows)


cache_loaded = False

if os.path.exists(EMD_CACHE_PATH):
    cached = pd.read_pickle(EMD_CACHE_PATH)

    required_keys = {'record_id', 'r_peak'}
    if required_keys.issubset(cached.columns) and len(cached) == len(df):
        same_records = np.array_equal(
            cached['record_id'].astype(str).values,
            df['record_id'].astype(str).values
        )
        same_peaks = np.array_equal(
            cached['r_peak'].values,
            df['r_peak'].values
        )

        if same_records and same_peaks:
            emd_feature_columns = [c for c in cached.columns if c.startswith('emd_')]
            emd_df = cached[emd_feature_columns].reset_index(drop=True)
            cache_loaded = True
            print(f'Loaded EMD features from cache: {EMD_CACHE_PATH}')

if not cache_loaded:
    print('Extracting EMD features...')
    emd_df = extract_all_emd_features(df)
    emd_df = emd_df.replace([np.inf, -np.inf], np.nan).fillna(0.0)

    cache_to_save = pd.concat([
        df[['record_id', 'r_peak']].reset_index(drop=True),
        emd_df.reset_index(drop=True)
    ], axis=1)
    cache_to_save.to_pickle(EMD_CACHE_PATH)
    print(f'Saved EMD cache to: {EMD_CACHE_PATH}')

print('EMD feature matrix shape:', emd_df.shape)
print('Number of EMD features:', len(emd_df.columns))

## 6. Leakage-safe train/test split by ECG record

`StratifiedGroupKFold` keeps all heartbeat windows from a given ECG record entirely in either training or testing while approximately preserving the class ratio.


In [ ]:
y = df['diabetes'].to_numpy(dtype=int)
groups = df['record_id'].astype(str).to_numpy()

splitter = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE
)

train_idx, test_idx = next(splitter.split(
    X=np.zeros(len(df)),
    y=y,
    groups=groups
))

y_train = y[train_idx]
y_test = y[test_idx]

train_groups = set(groups[train_idx])
test_groups = set(groups[test_idx])
assert train_groups.isdisjoint(test_groups), 'Group leakage detected!'

print(f'Train beat windows: {len(train_idx)}')
print(f'Test beat windows : {len(test_idx)}')
print(f'Train ECG records : {len(train_groups)}')
print(f'Test ECG records  : {len(test_groups)}')
print('\nTrain class counts:', np.bincount(y_train))
print('Test class counts :', np.bincount(y_test))

## 7. Build the final Sym4 + EMD feature matrix

There is **no ICA** in this version.

The classifier input contains:

- 47 EMD IMF/residue features
- 1 original beat-window standard-deviation feature

This gives 48 features when `MAX_IMFS = 5`.


In [ ]:
# EMD feature matrix
X_emd = emd_df.to_numpy(dtype=np.float64)

# Original beat-level statistical feature retained from the previous notebook
std_feature = df[['standard_deviation']].to_numpy(dtype=np.float64)

# Final Sym4 + EMD representation
X = np.hstack([X_emd, std_feature])
feature_names = list(emd_df.columns) + ['standard_deviation']

X_train = X[train_idx]
X_test  = X[test_idx]

print('Train feature matrix:', X_train.shape)
print('Test feature matrix :', X_test.shape)
print('Total feature count :', len(feature_names))
print()
print('First 10 feature names:')
print(feature_names[:10])


## 8. Train classifiers using Sym4 + EMD features

KNN and RBF-SVM are wrapped with `StandardScaler` because they are sensitive to feature scale. Tree-based models are trained directly on the EMD feature matrix.


In [ ]:
def binary_metrics(y_true, y_pred):
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()

    specificity = tn / (tn + fp) if (tn + fp) else 0.0

    return {
        'accuracy': accuracy_score(y_true, y_pred),
        'balanced_accuracy': balanced_accuracy_score(y_true, y_pred),
        'precision_diabetes': precision_score(y_true, y_pred, pos_label=1, zero_division=0),
        'sensitivity_recall': recall_score(y_true, y_pred, pos_label=1, zero_division=0),
        'specificity': specificity,
        'f1_diabetes': f1_score(y_true, y_pred, pos_label=1, zero_division=0),
    }


models = {
    'KNN': Pipeline([
        ('scaler', StandardScaler()),
        ('model', KNeighborsClassifier(n_neighbors=5))
    ]),

    'SVM (RBF)': Pipeline([
        ('scaler', StandardScaler()),
        ('model', SVC(
            kernel='rbf',
            C=1.0,
            gamma='scale',
            class_weight='balanced',
            random_state=RANDOM_STATE
        ))
    ]),

    'Decision Tree': DecisionTreeClassifier(
        criterion='gini',
        random_state=RANDOM_STATE,
        class_weight='balanced'
    ),

    'Random Forest': RandomForestClassifier(
        n_estimators=200,
        random_state=RANDOM_STATE,
        n_jobs=-1,
        class_weight='balanced_subsample'
    ),
}

trained_models = {}
predictions = {}
result_rows = []

for name, model in models.items():
    print()
    print(f'Training {name}...')
    model.fit(X_train, y_train)
    pred = model.predict(X_test)

    trained_models[name] = model
    predictions[name] = pred

    row = {'model': name}
    row.update(binary_metrics(y_test, pred))
    result_rows.append(row)

    print(classification_report(
        y_test,
        pred,
        target_names=['Normal', 'Diabetes'],
        zero_division=0
    ))

model_results = pd.DataFrame(result_rows).sort_values(
    'balanced_accuracy',
    ascending=False
).reset_index(drop=True)

print()
print('Sym4 + EMD model comparison:')
model_results


## 9. Confusion matrices


In [ ]:
for name, pred in predictions.items():
    cm = confusion_matrix(y_test, pred, labels=[0, 1])

    plt.figure(figsize=(6, 5))
    sns.heatmap(
        cm,
        annot=True,
        fmt='d',
        cmap='Blues',
        xticklabels=['Normal', 'Diabetes'],
        yticklabels=['Normal', 'Diabetes']
    )
    plt.xlabel('Predicted')
    plt.ylabel('True')
    plt.title(f'{name} — Sym4 + EMD')
    plt.tight_layout()
    plt.show()


## 10. Train-vs-test accuracy check

A large positive gap between training and test accuracy suggests overfitting.


In [ ]:
train_test_rows = []

for name, model in trained_models.items():
    train_pred = model.predict(X_train)
    test_pred = predictions[name]

    train_accuracy = accuracy_score(y_train, train_pred)
    test_accuracy = accuracy_score(y_test, test_pred)

    train_test_rows.append({
        'model': name,
        'train_accuracy': train_accuracy,
        'test_accuracy': test_accuracy,
        'generalization_gap': train_accuracy - test_accuracy
    })

train_test_results = pd.DataFrame(train_test_rows)
train_test_results


## 11. Feature significance on training data only

The Kruskal–Wallis test is performed only on training records so exploratory feature ranking does not inspect the held-out test records.


In [ ]:
train_feature_df = pd.DataFrame(
    X_train,
    columns=feature_names
)
train_feature_df['diabetes'] = y_train

kruskal_rows = []

for column in feature_names:
    diabetic_values = train_feature_df.loc[
        train_feature_df['diabetes'] == 1, column
    ]
    normal_values = train_feature_df.loc[
        train_feature_df['diabetes'] == 0, column
    ]

    # Skip a feature if both groups are constant at exactly the same value.
    combined = np.concatenate([diabetic_values.to_numpy(), normal_values.to_numpy()])
    if len(combined) == 0 or np.allclose(combined, combined[0]):
        stat, p_value = 0.0, 1.0
    else:
        stat, p_value = stats.kruskal(diabetic_values, normal_values)

    kruskal_rows.append({
        'feature': column,
        'H_statistic': stat,
        'p_value': p_value
    })

kruskal_results = pd.DataFrame(kruskal_rows).sort_values(
    'p_value'
).reset_index(drop=True)
kruskal_results['significant_p_lt_0_05'] = kruskal_results['p_value'] < 0.05

print('Top 25 Sym4 + EMD features by Kruskal-Wallis p-value:')
kruskal_results.head(25)


## 12. Correlation heatmap for selected EMD features


In [ ]:
selected = [
    c for c in [
        'emd_imf1_relative_energy',
        'emd_imf2_relative_energy',
        'emd_imf3_relative_energy',
        'emd_imf1_entropy',
        'emd_imf2_entropy',
        'emd_imf3_entropy',
        'emd_residue_energy',
        'emd_residue_std',
        'standard_deviation'
    ] if c in train_feature_df.columns
]
selected += ['diabetes']

corr = train_feature_df[selected].corr(numeric_only=True)

plt.figure(figsize=(11, 8))
sns.heatmap(corr, annot=True, cmap='coolwarm', vmin=-1, vmax=1, fmt='.2f')
plt.title('Training-set Correlation: Sym4 + EMD Features')
plt.tight_layout()
plt.show()


## 13. Class distribution and example ECG comparison


In [ ]:
label_counts = df['diabetes'].value_counts().sort_index()

plt.figure(figsize=(6, 6))
plt.pie(
    label_counts.values,
    labels=['Normal (0)', 'Diabetes (1)'],
    autopct='%1.1f%%',
    startangle=90,
    wedgeprops={'edgecolor': 'black'}
)
plt.title('Beat-window Label Distribution')
plt.show()


if len(diabetes_df) > 0 and len(normal_df) > 0:
    diabetes_signal = np.asarray(diabetes_df.iloc[0]['window_signal'])
    normal_signal = np.asarray(normal_df.iloc[0]['window_signal'])

    plt.figure(figsize=(10, 5))
    plt.plot(diabetes_signal, label='Diabetes ECG')
    plt.plot(normal_signal, label='Normal ECG')
    plt.legend()
    plt.title('Comparison of Sym4-denoised ECG Beat Windows')
    plt.xlabel('Samples')
    plt.ylabel('Normalized amplitude')
    plt.tight_layout()
    plt.show()

## 14. Random Forest feature importance

This provides a model-based view of which EMD-derived features contribute most strongly to the Random Forest classifier.


In [ ]:
rf_model = trained_models.get('Random Forest')

if rf_model is not None:
    importance_df = pd.DataFrame({
        'feature': feature_names,
        'importance': rf_model.feature_importances_
    }).sort_values('importance', ascending=False).reset_index(drop=True)

    display(importance_df.head(20))

    top_n = min(20, len(importance_df))
    top_importance = importance_df.head(top_n).sort_values('importance')

    plt.figure(figsize=(9, 7))
    plt.barh(top_importance['feature'], top_importance['importance'])
    plt.xlabel('Random Forest importance')
    plt.title('Top Sym4 + EMD Features')
    plt.tight_layout()
    plt.show()


## 15. Export Sym4 + EMD features for Power BI / later analysis

The exported table contains record identity, label, split membership, the original beat-window standard deviation, and all EMD-derived features. No ICA columns are present.


In [ ]:
export_df = pd.concat([
    df[['record_id', 'file', 'r_peak', 'diabetes', 'standard_deviation']].reset_index(drop=True),
    emd_df.reset_index(drop=True),
], axis=1)

export_df['split'] = 'train'
export_df.loc[test_idx, 'split'] = 'test'

output_csv = 'ecg_features_sym4_emd.csv'
export_df.to_csv(output_csv, index=False)

print('Saved:', output_csv)
print('Export shape:', export_df.shape)
print('ICA columns present:', any(c.startswith('ica_') for c in export_df.columns))

from google.colab import files
files.download(output_csv)


## Final method to report

The proposed method in this notebook is:

1. **Sym4 wavelet denoising** removes noise from the raw ECG signal.
2. The denoised signal is **Z-score normalized**.
3. **R-peaks** are detected and fixed 210-sample heartbeat windows are extracted.
4. **EMD** decomposes each heartbeat into up to five IMFs and a residue.
5. Statistical, energy, entropy, shape, and zero-crossing features are extracted from the IMFs.
6. ECG records are split into training and test groups with **no record overlap**.
7. The Sym4 + EMD feature vectors are classified using KNN, RBF-SVM, Decision Tree, and Random Forest.
8. Performance is evaluated using accuracy, balanced accuracy, precision, sensitivity, specificity, F1-score, and confusion matrices.

There is **no ICA stage** in this notebook.
